In [1]:
import sys
sys.path.append('../../Python_scripts')

## Import packages
from config import *
from support import *
from cosmo_support import *
import csv_to_latex as to_latex

/home/zhuge/miniconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
data=pd.read_csv('../../Data/FRB_new.csv')
print(data.columns)
data.head()

Index(['FRB', 'z', 'gl', 'gb', 'DM', 'DM_MW, ISM(NE2001)', 'DM_MW, ISM(YMW16)',
       'DM_ext(ne2001)', 'DM_ext(ymw16)', 'Ref.'],
      dtype='object')


,FRB,z,gl,gb,DM,"DM_MW, ISM(NE2001)","DM_MW, ISM(YMW16)",DM_ext(ne2001),DM_ext(ymw16),Ref.
0,FRB 20180814A,0.06800,136.460000,16.580000,190.90,87.83,108.41,73.073904,52.493323,CHIME13
1,FRB 20181030A,0.00385,133.400000,40.900000,103.50,40.35,32.24,33.149498,41.256729,181030A
2,FRB 20220529A,0.18390,130.787670,-41.858020,246.00,39.95,30.92,176.048832,185.076405,https://arxiv.org/abs/2503.04727
3,FRB 20220610A,1.01500,8.839200,-70.185700,1458.10,30.96,13.58,1397.142120,1414.521089,ASKAP2
4,FRB 20220717A,0.36295,19.835158,-17.632032,637.34,118.33,83.22,489.008396,524.116215,https://arxiv.org/pdf/2407.02173


In [3]:
data=data[['FRB', 'z','gl', 'gb','DM', 'DM_MW, ISM(NE2001)', 'DM_MW, ISM(YMW16)', 'Ref.']]
data.head()

,FRB,z,gl,gb,DM,"DM_MW, ISM(NE2001)","DM_MW, ISM(YMW16)",Ref.
0,FRB 20180814A,0.06800,136.460000,16.580000,190.90,87.83,108.41,CHIME13
1,FRB 20181030A,0.00385,133.400000,40.900000,103.50,40.35,32.24,181030A
2,FRB 20220529A,0.18390,130.787670,-41.858020,246.00,39.95,30.92,https://arxiv.org/abs/2503.04727
3,FRB 20220610A,1.01500,8.839200,-70.185700,1458.10,30.96,13.58,ASKAP2
4,FRB 20220717A,0.36295,19.835158,-17.632032,637.34,118.33,83.22,https://arxiv.org/pdf/2407.02173


In [4]:
import re
def frb_sort_key(frb_str):
    match = re.search(r'FRB\s*(\d+)([A-Z]+)', frb_str)
    if match:
        number = int(match.group(1))
        letter = match.group(2)
        return (number, letter)
    return (0, 'Z')
data_sorted = data.sort_values('FRB', key=lambda x: x.apply(frb_sort_key)).reset_index(drop=True)
data_sorted.head()

,FRB,z,gl,gb,DM,"DM_MW, ISM(NE2001)","DM_MW, ISM(YMW16)",Ref.
0,FRB 20121102A,0.19273,174.890,-0.230,557.00,188.42,287.07,https://iopscience.iop.org/article/10.1088/000...
1,FRB 20171020A,0.00867,29.300,-51.300,114.10,38.37,25.84,https://iopscience.iop.org/article/10.3847/204...
2,FRB 20180301A,0.33040,204.412,-6.481,522.00,150.73,252.56,https://iopscience.iop.org/article/10.3847/153...
3,FRB 20180814A,0.06800,136.460,16.580,190.90,87.83,108.41,CHIME13
4,FRB 20180916B,0.03380,129.710,3.730,348.76,198.91,324.82,180916B


In [5]:
data_sorted['gl']=data_sorted['gl'].round(4)
data_sorted['gb']=data_sorted['gb'].round(4)
data_sorted.head()

,FRB,z,gl,gb,DM,"DM_MW, ISM(NE2001)","DM_MW, ISM(YMW16)",Ref.
0,FRB 20121102A,0.19273,174.890,-0.230,557.00,188.42,287.07,https://iopscience.iop.org/article/10.1088/000...
1,FRB 20171020A,0.00867,29.300,-51.300,114.10,38.37,25.84,https://iopscience.iop.org/article/10.3847/204...
2,FRB 20180301A,0.33040,204.412,-6.481,522.00,150.73,252.56,https://iopscience.iop.org/article/10.3847/153...
3,FRB 20180814A,0.06800,136.460,16.580,190.90,87.83,108.41,CHIME13
4,FRB 20180916B,0.03380,129.710,3.730,348.76,198.91,324.82,180916B


In [6]:
citation_dict={
    '180916B':'2020Nature_180916, Tendulkar_2021_180916, Kaur_2022_180916',
    '181030A':'Bhardwaj_2021_181030A',
    'https://iopscience.iop.org/article/10.1088/0004-637X/790/2/101': '2017Nature_121102, Tendulkar_2017_121102', # FRB 121102
    'https://www.nature.com/articles/s41586-018-0864-x': '2019Nature_CHIME',
    'https://iopscience.iop.org/article/10.3847/2041-8213/ab4a80':'Andersen_2019',
    'https://arxiv.org/abs/2503.04727': '2025arXiv_YeLi',
    'ASKAP':'2025ASKAP',
    'ASKAP2':'2025ASKAP, highres_ASKAP_arxiv2025',
    'https://arxiv.org/pdf/2407.02173':'2024MNRAS_2trb_sys',
    'https://www.nature.com/articles/s41586-018-0588-y':'2018Nature171020',
    'https://academic.oup.com/mnras/article/486/3/3636/5432360?login=false':'2019MNRAS_180301',
    'https://www.science.org/doi/10.1126/science.aaw5903':'2019Science180924, 2020Nature_Macquart, Bhandari_2020, 2023ApJ_Garden, 2025ASKAP, highres_ASKAP_arxiv2025',# here
    'https://www.science.org/doi/10.1126/science.aay0073':'2019Science181112, 2020Nature_Macquart, Bhandari_2020, 2025ASKAP, highres_ASKAP_arxiv2025',
    'CHIME4':'Bhardwaj_2024',
    'CHIME_host':'Ibik_2024',
    'CHIME13':'Michilli_2023',
    '190608B':'2020Nature_Macquart, Bhandari_2020, 2023ApJ_Garden, 2025ASKAP, highres_ASKAP_arxiv2025',
    '290611B':'Heintz_2020, 2020Nature_Macquart, 2025ASKAP, highres_ASKAP_arxiv2025',
    '190711A':'Heintz_2020, 2020Nature_Macquart, 2025ASKAP, highres_ASKAP_arxiv2025',
    '190714A':'Heintz_2020, 2025ASKAP, highres_ASKAP_arxiv2025',
    '191001A':'Heintz_2020, 2025ASKAP, highres_ASKAP_arxiv2025',
    '191228A':'Bhandari_2022, 2025ASKAP, highres_ASKAP_arxiv2025',
    '200430A':'Heintz_2020, 2025ASKAP, highres_ASKAP_arxiv2025',
    '200906A':'Bhandari_2022, 2025ASKAP, highres_ASKAP_arxiv2025',
    '220501C':'2025ASKAP,Sharma, highres_ASKAP_arxiv2025',
    'https://www.nature.com/articles/s41586-022-04755-5':'2022Nature_CHNiu, 2023ApJ_Garden',
    'https://www.nature.com/articles/s41586-019-1389-7':'2019Nature_Ravi',
    'https://iopscience.iop.org/article/10.3847/1538-4357/aba4ac':'Law_2020',
    'https://iopscience.iop.org/article/10.3847/1538-3881/ac3aec':'Bhandari_2022',
    'https://academic.oup.com/mnras/article/514/2/1961/6595347?login=false':'2022MNRAS_Rajwade',
    'https://ui.adsabs.harvard.edu/abs/2022MNRAS.513..982R/abstract':'2022MNRAS_Ravi',
    'https://iopscience.iop.org/article/10.3847/1538-4357/acc178':'Bhandari_2023, 2023ApJ_Garden, 2025ASKAP, highres_ASKAP_arxiv2025',
    'https://arxiv.org/abs/2302.05465':'2023ApJ_Garden, 2025ASKAP, highres_ASKAP_arxiv2025',
    '20210405I':'2024MNRAS20210405I',
    'https://www.wis-tns.org/object/frb20210410d':'2023MNRAS.524.2064C, 2023ApJ_Garden',
    '210807D':'2023ApJ_Garden, 2025ASKAP',
    '211127I':'2023ApJ_Garden, Glowacki_2023, 2025ASKAP, highres_ASKAP_arxiv2025',
    '211203C':'2023ApJ_Garden, 2025ASKAP, highres_ASKAP_arxiv2025',
    '211212A':'2023ApJ_Garden, 2025ASKAP, highres_ASKAP_arxiv2025',
    '220105A':'2023ApJ_Garden, 2025ASKAP, highres_ASKAP_arxiv2025',
    'Kritti3':'2024ApJ_25,Sharma,Connor_Sharma', # Kritti3 24,28,29
    'Kritti4':'Sherman_2023,2024ApJ_25,Sharma,Connor_Sharma',
    'Kritti_local':'Sharma,Connor_Sharma',
    'Kritti2':'2024ApJ_25,Connor_Sharma', # Kritti2 24, 29
    'ASKAP_Kritti2':'Sharma,Connor_Sharma,2025ASKAP',
    'https://arxiv.org/abs/2307.09502':'2024NatAs...8.1429C',
    '20200120E':'Bhardwaj_2021, Glowacki_2023', # very small z in M81, change from 0.0008 to 0.00014
    'https://arxiv.org/pdf/2502.11217':'2025arXiv_CHIME_localized',
    'https://arxiv.org/pdf/2409.16952':'Connor_Sharma',
    '230521B':'Connor_Sharma,2025ASKAP',
    'https://iopscience.iop.org/article/10.3847/2041-8213/aae7cb':'Mahony_2018, 2023PASA_20171020A_revisited',
    '220319D':'Sherman_2023,2024ApJ_25,Sharma,Connor_Sharma, 2025AJ_20220319D',
    'https://www.astronomerstelegram.org/?read=15679':'2023ApJ_20220912A',
    '240114A':'2024ApJ_20240114A, Chen_2025',
    '190102C':'2020Nature_Macquart, Bhandari_2020, 2025ASKAP, highres_ASKAP_arxiv2025'
}

In [7]:
""" data_sorted['Ref.'] = data_sorted['Ref.'].astype(str)
data_sorted['Ref.'] = data_sorted['Ref.'].map(citation_dict).fillna(data_sorted['Ref.']) """

" data_sorted['Ref.'] = data_sorted['Ref.'].astype(str)\ndata_sorted['Ref.'] = data_sorted['Ref.'].map(citation_dict).fillna(data_sorted['Ref.']) "

In [8]:
data_sorted['Ref.'] = data_sorted['Ref.'].replace(citation_dict)
# data_sorted['Ref.']=r'\citet{'+data_sorted['Ref.']+'}'
data_sorted.head()

,FRB,z,gl,gb,DM,"DM_MW, ISM(NE2001)","DM_MW, ISM(YMW16)",Ref.
0,FRB 20121102A,0.19273,174.890,-0.230,557.00,188.42,287.07,"2017Nature_121102, Tendulkar_2017_121102"
1,FRB 20171020A,0.00867,29.300,-51.300,114.10,38.37,25.84,"Mahony_2018, 2023PASA_20171020A_revisited"
2,FRB 20180301A,0.33040,204.412,-6.481,522.00,150.73,252.56,Bhandari_2022
3,FRB 20180814A,0.06800,136.460,16.580,190.90,87.83,108.41,Michilli_2023
4,FRB 20180916B,0.03380,129.710,3.730,348.76,198.91,324.82,"2020Nature_180916, Tendulkar_2021_180916, Kaur..."


In [9]:
print(data_sorted.columns)
data_sorted.rename(columns={
    'DM': r'$\DM_{\rm obs}$',
    'DM_MW, ISM(NE2001)': r'$\DM_{\rm MW, ISM}$(NE2001)',
    'DM_MW, ISM(YMW16)': r'$\DM_{\rm MW, ISM}$(YMW16)',
}, inplace=True)
data_sorted.head()

Index(['FRB', 'z', 'gl', 'gb', 'DM', 'DM_MW, ISM(NE2001)', 'DM_MW, ISM(YMW16)',
       'Ref.'],
      dtype='object')


,FRB,z,gl,gb,$\DM_{\rm obs}$,"$\DM_{\rm MW, ISM}$(NE2001)","$\DM_{\rm MW, ISM}$(YMW16)",Ref.
0,FRB 20121102A,0.19273,174.890,-0.230,557.00,188.42,287.07,"2017Nature_121102, Tendulkar_2017_121102"
1,FRB 20171020A,0.00867,29.300,-51.300,114.10,38.37,25.84,"Mahony_2018, 2023PASA_20171020A_revisited"
2,FRB 20180301A,0.33040,204.412,-6.481,522.00,150.73,252.56,Bhandari_2022
3,FRB 20180814A,0.06800,136.460,16.580,190.90,87.83,108.41,Michilli_2023
4,FRB 20180916B,0.03380,129.710,3.730,348.76,198.91,324.82,"2020Nature_180916, Tendulkar_2021_180916, Kaur..."


In [10]:
head2=r'& & (deg) & (deg) & $ (\rm pc \, cm^{-3})$ & $(\rm pc \, cm^{-3})$ & $(\rm pc \, cm^{-3})$ & '
to_latex.dataframe_to_latex_long_ref(data_sorted,'./table.tex',caption='Well localized FRB samples',label='tab:data',head2=head2)

LaTeX already save to: ./table.tex


'\\startlongtable\n\\begin{deluxetable}{cccccccc}\n\\label{tab:data}\n\\centering\n\\tablecaption{Well localized FRB samples}\n\\tablehead{\\colhead{FRB} & \\colhead{z} & \\colhead{gl} & \\colhead{gb} & \\colhead{$\\DM_{\\rm obs}$} & \\colhead{$\\DM_{\\rm MW, ISM}$(NE2001)} & \\colhead{$\\DM_{\\rm MW, ISM}$(YMW16)} & \\colhead{Ref.} \\\\\n& & (deg) & (deg) & $ (\\rm pc \\, cm^{-3})$ & $(\\rm pc \\, cm^{-3})$ & $(\\rm pc \\, cm^{-3})$ & \n}\n\\startdata\nFRB 20121102A & 0.19273 & 174.89 & -0.23 & 557.0 & 188.42 & 287.07 & 1, 40 \\\\\nFRB 20171020A & 0.00867 & 29.3 & -51.3 & 114.1 & 38.37 & 25.84 & 36, 13 \\\\\nFRB 20180301A & 0.3304 & 204.412 & -6.481 & 522.0 & 150.73 & 252.56 & 24 \\\\\nFRB 20180814A & 0.068 & 136.46 & 16.58 & 190.9 & 87.83 & 108.41 & 37 \\\\\nFRB 20180916B & 0.0338 & 129.71 & 3.73 & 348.76 & 198.91 & 324.82 & 5, 41, 34 \\\\\nFRB 20180924B & 0.3214 & 0.7424 & -49.4147 & 361.75 & 40.5 & 27.65 & 3, 6, 23, 11, 20, 42 \\\\\nFRB 20181030A & 0.00385 & 133.4 & 40.9 & 103.5 & 